## Install Dependencies

In [1]:
!pip install GEOparse bioservices mygene

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.2/45.2 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 258.0/258.0 kB 16.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.7/46.7 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.0/57.0 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.4/61.4 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.2/70.2 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 145.3/145.3 kB 12.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.0/88.0 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.7/70.7 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 720.1/720.1 kB 42.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 68.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 264.7/264.7 kB 19.8 MB/s eta 0:00:00
  Attempting uninstall: wrapt
    Found existing in

In [18]:
import numpy as np
import pandas as pd

from bioservices import KEGG
import re
import random
import GEOparse
from itertools import islice
from collections import defaultdict, Counter

## GSE10245 Dataset from GEO

https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE48558

In [3]:
# Download and parse the dataset
gse = GEOparse.get_GEO("GSE10245", destdir=".", silent=True)

/usr/local/lib/python3.12/dist-packages/GEOparse/GEOparse.py:401: DtypeWarning: Columns (2) have mixed types. Specify dtype option on import or set low_memory=False.
  return read_csv(StringIO(data), index_col=None, sep="\t")


In [4]:
print(gse)

<SERIES: GSE10245 - 58 SAMPLES, 1 d(s)>


In [5]:
# List all sample IDs
print(gse.gsms.keys())

# Inspect the first sample’s metadata
first_sample = list(gse.gsms.values())[0]
print(first_sample.metadata)

dict_keys(['GSM258551', 'GSM258552', 'GSM258553', 'GSM258554', 'GSM258555', 'GSM258556', 'GSM258557', 'GSM258558', 'GSM258559', 'GSM258560', 'GSM258561', 'GSM258562', 'GSM258563', 'GSM258564', 'GSM258565', 'GSM258566', 'GSM258567', 'GSM258568', 'GSM258569', 'GSM258570', 'GSM258571', 'GSM258572', 'GSM258573', 'GSM258574', 'GSM258575', 'GSM258576', 'GSM258577', 'GSM258578', 'GSM258579', 'GSM258580', 'GSM258581', 'GSM258582', 'GSM258583', 'GSM258584', 'GSM258585', 'GSM258586', 'GSM258587', 'GSM258588', 'GSM258589', 'GSM258590', 'GSM258591', 'GSM258592', 'GSM258593', 'GSM258594', 'GSM258595', 'GSM258596', 'GSM258597', 'GSM258598', 'GSM258599', 'GSM258600', 'GSM258601', 'GSM258602', 'GSM258603', 'GSM258604', 'GSM258605', 'GSM258606', 'GSM258607', 'GSM258608'])
{'title': ['NSCLC_AC_10'], 'geo_accession': ['GSM258551'], 'status': ['Public on Oct 01 2009'], 'submission_date': ['Jan 23 2008'], 'last_update_date': ['Aug 28 2018'], 'type': ['RNA'], 'channel_count': ['1'], 'source_name_ch1': ['hum

In [10]:
meta_rows = []
for gsm_name, gsm in gse.gsms.items():
    src = gsm.metadata.get("source_name_ch1", None)
    if src:
        source_name = src[0]
    else:
        title = gsm.metadata.get("title", [""])[0]
        source_name = title or "Unknown"

    # find disease state entry inside characteristics_ch1
    chars = gsm.metadata.get("characteristics_ch1", [])
    disease_state = None

    for c in chars:
        if isinstance(c, str) and "disease state" in c.lower():
            parts = c.split(":", 1)
            if len(parts) == 2:
                disease_state = parts[1].strip().lower()
            else:
                disease_state = c.strip().lower()
            break

    meta_rows.append({
        "sample_id": gsm_name,
        "source_name": source_name,
        "disease_state": disease_state
    })

meta_df = pd.DataFrame(meta_rows).set_index("sample_id")
print("Total samples in metadata:", meta_df.shape[0])

Total samples in metadata: 58


In [11]:
print(meta_df["disease_state"].value_counts())

disease_state
adenocarcinoma             40
squamous cell carcinoma    18
Name: count, dtype: int64


In [12]:
# build expression dataframe (samples x probes)
expr_df = pd.DataFrame({
    gsm_name: gsm.table.set_index("ID_REF")["VALUE"].astype(float)
    for gsm_name, gsm in gse.gsms.items()
}).transpose()

print("Total samples in expr_df:", expr_df.shape[0], "probes:", expr_df.shape[1])

Total samples in expr_df: 58 probes: 54675


In [13]:
# Map recurrence classes to numeric labels
label_map = {
    "adenocarcinoma": 0,
    "squamous cell carcinoma": 1
}

# Create label series aligned with expr_df index
labels = meta_df.loc[expr_df.index, "disease_state"].map(label_map)

# Add label column as the last column
expr_df_labeled = expr_df.assign(label=labels.values)

# Ensure label is int
expr_df_labeled["label"] = expr_df_labeled["label"].astype(int)

print(expr_df_labeled.shape)
expr_df_labeled.head(3)

(58, 54676)


ID_REF,1007_s_at,1053_at,117_at,121_at,1255_g_at,1294_at,1316_at,1320_at,1405_i_at,1431_at,...,AFFX-r2-Ec-bioB-3_at,AFFX-r2-Ec-bioB-5_at,AFFX-r2-Ec-bioB-M_at,AFFX-r2-Ec-bioC-3_at,AFFX-r2-Ec-bioC-5_at,AFFX-r2-Ec-bioD-3_at,AFFX-r2-Ec-bioD-5_at,AFFX-r2-P1-cre-3_at,AFFX-r2-P1-cre-5_at,label
GSM258551,9.129905,8.034022,3.564520,4.746490,2.320698,5.519153,3.339182,2.775395,8.303437,2.981327,...,8.857847,9.369037,9.435077,10.936404,10.730983,13.599488,13.031726,15.028729,14.586347,0
GSM258552,9.843349,7.973332,4.994852,5.197306,2.248520,5.081258,2.934516,2.617097,9.145519,4.619668,...,7.865127,8.116953,8.306683,9.986505,9.719266,12.847711,12.250033,14.440756,14.072366,0
GSM258553,9.730661,8.834045,5.066018,5.234618,2.259504,4.657257,3.007192,2.634559,9.159054,2.746464,...,6.996104,7.803869,8.029131,9.468096,9.101115,12.384142,11.798363,14.439887,14.011392,1


In [14]:
expr_df_labeled.to_csv("original.csv", index=True)
print(f"Saved to csv.")

Saved to csv.


## Map probe -> entrez -> gene symbol

In [16]:
def fetch_gpl_table(gpl_acc=None, destdir="."):
    print(f"Downloading {gpl_acc} ...")
    gpl = GEOparse.get_GEO(geo=gpl_acc, destdir=destdir, silent=True)
    table = gpl.table
    print("GPL downloaded. Table shape:", table.shape)
    return table

def build_probe2symbol_from_table(table, probe_col, symbol_col):
    mapping = {}
    missing = 0
    for idx, row in table.iterrows():
        probe = str(row[probe_col]).strip()
        symbol_raw = row.get(symbol_col, "")
        if pd.isna(symbol_raw):
            mapping[probe] = None
            missing += 1
            continue
        symbol = str(symbol_raw).strip()
        # Clean symbol: some GPLs have 'SYMBOL /// Another' or 'GENE: SYMBOL (desc)'
        # try a few heuristics:
        # 1) split on '///' or '///' or ' // '
        symbol = re.split(r"\s*///\s*|\s*//\s*|;", symbol)[0]
        # 2) if the token contains spaces and likely is description, try to extract first token that looks like a gene symbol
        tokens = re.split(r"[,\s\(\)]+", symbol)
        # keep first token that matches typical gene symbol pattern (letters, digits, dash)
        sym_candidate = None
        for t in tokens:
            if re.match(r"^[A-Za-z0-9\-\._]+$", t):
                # skip tokens that are '---' or 'NA' or numeric only
                if t.upper() in ("NA", "---"):
                    continue
                if re.search(r"[A-Za-z]", t):
                    sym_candidate = t
                    break
        if sym_candidate is None:
            sym_candidate = symbol  # fallback to raw cleaned string
        mapping[probe] = sym_candidate
    print(f"Built mapping for {len(mapping)} probes ({missing} missing symbols).")
    return mapping

In [17]:
gpl_table = fetch_gpl_table("GPL570")
probe_col = "ID"
symbol_col = "Gene Symbol"

GPL downloaded. Table shape: (54675, 16)


/usr/local/lib/python3.12/dist-packages/GEOparse/GEOparse.py:401: DtypeWarning: Columns (2) have mixed types. Specify dtype option on import or set low_memory=False.
  return read_csv(StringIO(data), index_col=None, sep="\t")


In [19]:
probe2symbol = build_probe2symbol_from_table(gpl_table, probe_col, symbol_col)

print("Example of probe2symbol: ")
example1 = dict(islice(probe2symbol.items(), 5))
example1

Built mapping for 54675 probes (8893 missing symbols).
Example of probe2symbol: 


{'1007_s_at': 'DDR1',
 '1053_at': 'RFC2',
 '117_at': 'HSPA6',
 '121_at': 'PAX8',
 '1255_g_at': 'GUCA1A'}

In [20]:
dataset_genes = list(set(probe2symbol.values()))
print("Number of unique symbols:", len(dataset_genes))

Number of unique symbols: 22881


In [21]:
def map_col2gene_df(df, probe2symbol, label_col):

    df = df.copy()

    # identify probe columns in the dataframe that exist in probe2symbol
    probes_in_df = [c for c in df.columns if c in probe2symbol]
    # drop probes with missing symbol mapping
    probes_with_symbol = [p for p in probes_in_df if probe2symbol.get(p) is not None]

    # compose subset columns to keep
    keep_cols = probes_with_symbol[:]
    keep_cols.append(label_col)

    df = df.loc[:, keep_cols]

    # rename probes to gene symbols
    rename_map = {probe: probe2symbol[probe] for probe in probes_with_symbol}
    df = df.rename(columns=rename_map)

    # collapse probes that map to the same symbol by mean
    expr = df.drop(columns=[label_col])
    cancer_series = df[label_col]
    expr = expr.groupby(expr.columns, axis=1).mean()

    # reattach cancer column
    final_df = expr.copy()
    final_df[label_col] = cancer_series.reindex(final_df.index)

    return final_df

In [22]:
gene_symbol_df = map_col2gene_df(expr_df_labeled, probe2symbol, label_col="label")

print(gene_symbol_df.shape)
gene_symbol_df.head(5)

(58, 22881)


/tmp/ipython-input-162475842.py:23: FutureWarning: DataFrame.groupby with axis=1 is deprecated. Do `frame.T.groupby(...)` without axis instead.
  expr = expr.groupby(expr.columns, axis=1).mean()


ID_REF,A1BG,A1BG-AS1,A1CF,A2M,A2M-AS1,A2ML1,A2MP1,A4GALT,A4GNT,AA06,...,ZYG11A,ZYG11B,ZYX,ZZEF1,ZZZ3,abParts,av27s1,hsa-let-7a-3,mir-223,label
GSM258551,3.423433,3.205138,2.857088,7.672613,3.502342,2.761913,2.296852,2.807237,2.258166,2.813350,...,2.429069,7.835881,8.338647,4.486576,6.500031,12.820440,2.369786,2.383189,2.397834,0
GSM258552,4.576181,3.047956,2.557988,7.272212,3.582006,2.452546,2.200790,3.170852,2.233875,2.732083,...,2.937927,7.848085,8.159407,4.666272,7.079585,14.437977,2.305893,2.315745,4.233613,0
GSM258553,3.251567,2.926385,2.658394,7.124156,3.505555,2.684441,2.233857,5.564696,2.256392,2.905625,...,4.396966,7.808464,7.664800,4.257710,6.769417,12.492411,2.373845,2.347268,2.057785,1
GSM258554,3.418560,2.989130,2.716686,7.034452,3.508446,2.565077,2.254745,2.812602,2.282962,2.806266,...,2.232935,8.546458,8.161060,4.330382,7.268318,13.211087,2.388046,2.352210,2.446505,0
GSM258555,3.342006,2.886925,2.566067,6.219881,3.587962,4.078021,2.204716,3.458652,2.218801,2.722117,...,2.200520,7.905053,8.100753,4.251840,7.015165,6.984485,2.298069,2.302594,2.629158,1


In [23]:
gene_symbol_df.to_csv("original_mapped.csv", index=True)
print("saved csv")

saved csv


In [24]:
gene_symbols = {c for c in gene_symbol_df.columns if c != "label"}

## Get genes from KEGG cancer pathways

In [25]:
unique_intersect_gene = set()

In [26]:
def fetch_kegg_aml_symbols(pathway):
    k = KEGG()
    entry = k.get(pathway)
    parsed = k.parse(entry)
    symbols = set()
    if 'GENE' in parsed:
        for entrez, desc in parsed['GENE'].items():
            m = re.match(r"^([A-Za-z0-9\-\._]+)", desc)
            if m:
                symbols.add(m.group(1).upper().strip())
    return symbols

def add_to_set(list):
    for item in list:
      if item not in unique_intersect_gene:
        unique_intersect_gene.add(item)
      pass

def get_intersect_genes(gene_symbols, pathway):
    symbols_in_pathway = fetch_kegg_aml_symbols(pathway)
    print(f'Number of symbols in {pathway}: {len(symbols_in_pathway)}')

    intersected_genes = sorted(list(gene_symbols.intersection(symbols_in_pathway)))
    print("Matched genes:", len(intersected_genes))

    add_to_set(intersected_genes)
    print("Number of intersected gene: ", len(unique_intersect_gene))

In [27]:
kegg_pathway_list = [
    'hsa05200',
    'hsa05205',
    'hsa05206',
    'hsa05202',
    'hsa05223',
    'hsa05230',
    'hsa05222',
    'hsa05224',
    'hsa05215',
    'hsa05213',
    'hsa05226',
    'hsa05210',
    'hsa05212',
    'hsa05216',
    'hsa05219',
    'hsa05231',
    'hsa05235',
    'hsa05217',
    'hsa05208',
    'hsa05203',
    'hsa05169',
    'hsa05211',
    'hsa05204',
    'hsa05225',
    'hsa05207',
    'hsa05218',
    'hsa05214',
    'hsa05165',
    'hsa04371',
    'hsa03460',
    'hsa03320'
]

In [28]:
for i in kegg_pathway_list:
  get_intersect_genes(gene_symbols, i)

Creating directory /root/.cache/bioservices 
Welcome to Bioservices
It looks like you do not have a configuration file.
We are creating one with default values in /root/.config/bioservices/bioservices.cfg .
Done
Number of symbols in hsa05200: 533
Matched genes: 517
Number of intersected gene:  517
Number of symbols in hsa05205: 204
Matched genes: 197
Number of intersected gene:  605
Number of symbols in hsa05206: 320
Matched genes: 171
Number of intersected gene:  686
Number of symbols in hsa05202: 201
Matched genes: 171
Number of intersected gene:  795
Number of symbols in hsa05223: 73
Matched genes: 72
Number of intersected gene:  800
Number of symbols in hsa05230: 71
Matched genes: 67
Number of intersected gene:  830
Number of symbols in hsa05222: 93
Matched genes: 92
Number of intersected gene:  830
Number of symbols in hsa05224: 148
Matched genes: 144
Number of intersected gene:  835
Number of symbols in hsa05215: 106
Matched genes: 104
Number of intersected gene:  856
Number of s

In [29]:
with open("kegg_intersect.txt", "w") as f:
  f.write(repr(unique_intersect_gene))

## Extract subset from intersected genes

In [30]:
df = pd.read_csv('original_mapped.csv')
df.head(5)

,Unnamed: 0,A1BG,A1BG-AS1,A1CF,A2M,A2M-AS1,A2ML1,A2MP1,A4GALT,A4GNT,...,ZYG11A,ZYG11B,ZYX,ZZEF1,ZZZ3,abParts,av27s1,hsa-let-7a-3,mir-223,label
0,GSM258551,3.423433,3.205138,2.857088,7.672613,3.502342,2.761913,2.296852,2.807237,2.258166,...,2.429069,7.835881,8.338647,4.486576,6.500031,12.820440,2.369786,2.383189,2.397834,0
1,GSM258552,4.576181,3.047956,2.557988,7.272212,3.582006,2.452546,2.200790,3.170852,2.233875,...,2.937927,7.848085,8.159407,4.666272,7.079585,14.437977,2.305893,2.315745,4.233613,0
2,GSM258553,3.251567,2.926385,2.658394,7.124156,3.505555,2.684441,2.233857,5.564696,2.256392,...,4.396966,7.808464,7.664800,4.257710,6.769417,12.492411,2.373845,2.347268,2.057785,1
3,GSM258554,3.418560,2.989130,2.716686,7.034452,3.508446,2.565077,2.254745,2.812602,2.282962,...,2.232935,8.546458,8.161060,4.330382,7.268318,13.211087,2.388046,2.352210,2.446505,0
4,GSM258555,3.342006,2.886925,2.566067,6.219881,3.587962,4.078021,2.204716,3.458652,2.218801,...,2.200520,7.905053,8.100753,4.251840,7.015165,6.984485,2.298069,2.302594,2.629158,1


In [31]:
with open("kegg_intersect.txt") as f:
    initial_intersect = eval(f.read())

print(len(initial_intersect))
print(list(initial_intersect)[:10])

1577
['BCL11B', 'FANCB', 'BCL6', 'ITGA3', 'NR4A3', 'IFNA10', 'COX4I1', 'THBS4', 'SMO', 'SLC25A4']


In [32]:
df.columns = [str(c).strip() for c in df.columns]
if "Unnamed: 0" in df.columns:
    df = df.drop(columns=["Unnamed: 0"])

In [33]:
df.head()

,A1BG,A1BG-AS1,A1CF,A2M,A2M-AS1,A2ML1,A2MP1,A4GALT,A4GNT,AA06,...,ZYG11A,ZYG11B,ZYX,ZZEF1,ZZZ3,abParts,av27s1,hsa-let-7a-3,mir-223,label
0,3.423433,3.205138,2.857088,7.672613,3.502342,2.761913,2.296852,2.807237,2.258166,2.813350,...,2.429069,7.835881,8.338647,4.486576,6.500031,12.820440,2.369786,2.383189,2.397834,0
1,4.576181,3.047956,2.557988,7.272212,3.582006,2.452546,2.200790,3.170852,2.233875,2.732083,...,2.937927,7.848085,8.159407,4.666272,7.079585,14.437977,2.305893,2.315745,4.233613,0
2,3.251567,2.926385,2.658394,7.124156,3.505555,2.684441,2.233857,5.564696,2.256392,2.905625,...,4.396966,7.808464,7.664800,4.257710,6.769417,12.492411,2.373845,2.347268,2.057785,1
3,3.418560,2.989130,2.716686,7.034452,3.508446,2.565077,2.254745,2.812602,2.282962,2.806266,...,2.232935,8.546458,8.161060,4.330382,7.268318,13.211087,2.388046,2.352210,2.446505,0
4,3.342006,2.886925,2.566067,6.219881,3.587962,4.078021,2.204716,3.458652,2.218801,2.722117,...,2.200520,7.905053,8.100753,4.251840,7.015165,6.984485,2.298069,2.302594,2.629158,1


In [34]:
# keep only intersected genes that are present in the df
genes_present = [g for g in initial_intersect if g in df.columns]
missing_genes = [g for g in initial_intersect if g not in df.columns]
cols_to_keep = genes_present + ["label"]

print(f"Sampled genes found in dataframe: {len(genes_present)}")

Sampled genes found in dataframe: 1577


In [35]:
reduced_df = df.loc[:, cols_to_keep].copy()

print("Reduced dataframe shape (rows x cols):", reduced_df.shape)
print("Label distribution in reduced set:")
print(reduced_df["label"].value_counts())

Reduced dataframe shape (rows x cols): (58, 1578)
Label distribution in reduced set:
label
0    40
1    18
Name: count, dtype: int64


In [36]:
reduced_df.to_csv("reduced.csv", index=False)
print(f"Saved reduced dataframe")

Saved reduced dataframe
